# Carbon-Budgeted Evaluation — CIFAR-10

**How to use:**
1. Run all cells in order.
2. To switch models, change `MODEL_NAME` in the *Model Selection* cell to one of:
   `"lenet5"`, `"vgg16"`, `"resnet18"`, `"resnet50"`, `"mobilenetv2"`, `"efficientnetb0"`
3. Results are saved to `BASE_RESULTS_DIR/epochwise_carbon_{MODEL_NAME}.csv`.

In [ ]:
# ── Install dependencies (skip if already installed) ──────────────────────────
import importlib, subprocess, sys

if importlib.util.find_spec('codecarbon') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'codecarbon'])

In [ ]:
# ── Path setup (works locally AND on Google Colab) ────────────────────────────
import os, sys

# In Jupyter, the kernel cwd is always the notebook's directory.
# This is the most reliable cross-environment way to locate models/.
NOTEBOOK_DIR = os.getcwd()
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

# ── On Colab: mount Drive if needed ──────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    # Point NOTEBOOK_DIR to the cifar-10 folder inside your cloned repo on Drive.
    # Adjust REPO_PATH if your Drive folder is named differently.
    REPO_PATH    = '/content/drive/MyDrive/green-ai-cnn-carbon'
    NOTEBOOK_DIR = os.path.join(REPO_PATH, 'Carbon_budgeting', 'cifar-10')
    if NOTEBOOK_DIR not in sys.path:
        sys.path.insert(0, NOTEBOOK_DIR)
except ImportError:
    ON_COLAB = False

print(f'Running on Colab : {ON_COLAB}')
print(f'NOTEBOOK_DIR     : {NOTEBOOK_DIR}')
print(f'sys.path[0]      : {sys.path[0]}')

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import tensorflow as tf
import numpy as np
import pandas as pd
import time

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from codecarbon import EmissionsTracker

# ── Import model registry from models/ ───────────────────────────────────────
from models import get_model, preprocess_data

print('TensorFlow:', tf.__version__)
print('Available models: lenet5, vgg16, resnet18, resnet50, mobilenetv2, efficientnetb0')

In [ ]:
# ── Results directory ─────────────────────────────────────────────────────────
# Saved relative to this notebook. On Colab you can override with a Drive path:
#   BASE_RESULTS_DIR = '/content/drive/MyDrive/green-ai-cnn-carbon/results/cifar10'
BASE_RESULTS_DIR = os.path.join(NOTEBOOK_DIR, 'results', 'cifar10')
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)
print('Results dir:', BASE_RESULTS_DIR)

In [ ]:
# ── Data loading — standard CIFAR-10 normalisation to [0, 1] ─────────────────
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test,  10)

print(f'Train: {x_train.shape}  Test: {x_test.shape}')

In [ ]:
# ── Model Selection ── change MODEL_NAME to switch models ─────────────────────
# Options: 'lenet5' | 'vgg16' | 'resnet18' | 'resnet50' | 'mobilenetv2' | 'efficientnetb0'
MODEL_NAME  = 'lenet5'
EPOCHS      = 50
BATCH_SIZE  = 64
NUM_CLASSES = 10

# Apply model-specific preprocessing.
# lenet5/vgg16/resnet18/resnet50 → identity (already [0,1])
# mobilenetv2                    → preprocess_input(x*255) → [-1, 1]
# efficientnetb0                 → preprocess_input(x*255) → Keras internal scaling
x_train_m, x_test_m = preprocess_data(MODEL_NAME, x_train, x_test)

# Build & compile model
model = get_model(MODEL_NAME, num_classes=NUM_CLASSES)
model.summary()

In [ ]:
# ── Carbon-tracked per-epoch training loop ────────────────────────────────────
TRACKING_DIR = os.path.join(NOTEBOOK_DIR, 'tracking')
os.makedirs(TRACKING_DIR, exist_ok=True)

tracker = EmissionsTracker(
    project_name=f'carbon_budget_{MODEL_NAME}_cifar10',
    output_dir=TRACKING_DIR,
    log_level='error',
)

epoch_logs = []
tracker.start()
start_time = time.time()

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')

    model.fit(
        x_train_m, y_train,
        batch_size=BATCH_SIZE,
        epochs=1,
        verbose=1,
    )

    loss, acc = model.evaluate(x_test_m, y_test, verbose=0)

    emissions_kg = tracker.flush()
    emissions_g  = emissions_kg * 1000
    elapsed_min  = (time.time() - start_time) / 60

    epoch_logs.append({
        'dataset':          'CIFAR-10',
        'model':            MODEL_NAME,
        'epoch':            epoch + 1,
        'val_accuracy':     acc,
        'cumulative_co2_g': emissions_g,
        'elapsed_time_min': elapsed_min,
    })

    print(f'  Val Acc: {acc:.4f} | CO\u2082: {emissions_g:.4f} g | Time: {elapsed_min:.1f} min')

tracker.stop()

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
df = pd.DataFrame(epoch_logs)
csv_path = os.path.join(BASE_RESULTS_DIR, f'epochwise_carbon_{MODEL_NAME}.csv')
df.to_csv(csv_path, index=False)
print(f'Saved \u2192 {csv_path}')
df.head()

In [ ]:
# ── Carbon-budgeted accuracy summary ─────────────────────────────────────────
# Run after training all models. Merges all per-model CSVs and computes
# the best accuracy achievable within each carbon budget.
import glob

all_csvs = glob.glob(os.path.join(BASE_RESULTS_DIR, 'epochwise_carbon_*.csv'))
df_all = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)

CARBON_BUDGETS = [1, 5, 10]  # grams CO\u2082
results = []

for mdl in df_all['model'].unique():
    df_m = df_all[df_all['model'] == mdl]
    for B in CARBON_BUDGETS:
        df_budget = df_m[df_m['cumulative_co2_g'] <= B]
        acc_B = df_budget['val_accuracy'].max() if not df_budget.empty else 0
        results.append({'model': mdl, 'carbon_budget_g': B, 'carbon_budgeted_accuracy': acc_B})

df_budgeted = pd.DataFrame(results)
pivot = df_budgeted.pivot(index='model', columns='carbon_budget_g', values='carbon_budgeted_accuracy')
pivot_path = os.path.join(NOTEBOOK_DIR, 'budgeted_accuracy_pivot.csv')
pivot.to_csv(pivot_path)
print(f'Pivot saved \u2192 {pivot_path}')
pivot